In [ ]:
import faiss
import pickle
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# load the index here
bert_index = faiss.read_index("REPLACE W INDEX FILE NAME")

with open("NAME OF YOUR METADATA FILE DOES HERE", "rb") as f:
    jobs = pickle.load(f)

bert = HuggingFaceEmbeddings(model_name="google-bert/bert-base-uncased")

def search_jobs(query, location_filter=None, top_k=10):
    """Search for jobs in FAISS and filter by location."""
    query_vector = np.array([bert.embed_query(query)], dtype=np.float32)
    D, I = bert_index.search(query_vector, k=top_k)
    retrieved_jobs = [jobs[idx] for idx in I[0] if idx != -1]
    if location_filter:
        retrieved_jobs = [
            job for job in retrieved_jobs if location_filter.lower() in job["Location"].lower()
        ]
    return retrieved_jobs

# example query
query = "software engineer"
location = "Irvine"

results = search_jobs(query, location_filter=location, top_k=10)
if results:
    print(f"Top {len(results)} jobs in {location} for query: '{query}':\n")
    for job in results:
        print(f"Title: {job['Title']}\nLocation: {job['Location']}\nDescription: {job['Description'][:200]}...\n")
else:
    print(f"No jobs found in {location} for query: '{query}'")